# 🧭 **ML Lab 7 — Scaling, Choosing k, and Pipelines**

Last lab, we built a model. Today, we improve that same model — this is not a new project, it's a continuation of the Loan Approval work from Lab 6.

You already know every tool used today (`ColumnTransformer`, `OneHotEncoder`, `StandardScaler`, `KNeighborsClassifier`, `.fit()`, `.predict()`, `accuracy_score`) from Lab 6. Today you'll use them a little differently, and you'll **write the code yourself** — small pieces, one concept at a time.

`"passthrough"` is the only new keyword today — it tells a `ColumnTransformer` to leave a column exactly as it is, with no scaling applied.

---



# ⚙️ **0. Setup — Rebuilding Lab 6: Loan Approval**

### _This is exactly the Loan Approval pipeline from Lab 6 — run it once and reuse every variable it creates for the rest of this lab. You do not need to write this again._

---



In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

loan_df = pd.read_csv('loan_approval.csv')
loan_df = loan_df.drop(columns=['applicant_id'])
loan_df['credit_score'] = loan_df['credit_score'].fillna(loan_df['credit_score'].median())
loan_df['years_at_job'] = loan_df['years_at_job'].fillna(loan_df['years_at_job'].median())

loan_X = loan_df.drop(columns=['approved'])
loan_y = loan_df['approved']

loan_cat_cols = ['employment_type', 'city_tier']
loan_num_cols = ['age', 'annual_income', 'loan_amount_lakh', 'credit_score', 'existing_loans', 'years_at_job']

loan_X_train, loan_X_test, loan_y_train, loan_y_test = train_test_split(
    loan_X, loan_y, test_size=0.2, random_state=42
)

loan_ct = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore'), loan_cat_cols),
    ('num', StandardScaler(), loan_num_cols)
])

loan_X_train_t = loan_ct.fit_transform(loan_X_train)
loan_X_test_t = loan_ct.transform(loan_X_test)

loan_knn = KNeighborsClassifier(n_neighbors=5)
loan_knn.fit(loan_X_train_t, loan_y_train)
loan_baseline_acc = accuracy_score(loan_y_test, loan_knn.predict(loan_X_test_t))
print('Loan Approval baseline accuracy (k=5, scaled):', loan_baseline_acc)


# 🏦 **Part 1 — Loan Approval (Guided Classroom Practice)**

### _Three small improvements to the SAME model above — not three separate programs. Q1.1 changes the preprocessing, Q1.2 reuses it to pick a better `k`, and Q1.3 reuses both to build one clean `Pipeline`._

---



## **Q1.1 — Scaling OFF**

**Concept:** KNN decides its prediction using *distance* between points. What happens to that distance if the numeric columns are not scaled?

**Your task:**
1. Build a new `ColumnTransformer` called `loan_ct_nosc` — keep the same `OneHotEncoder` step for `loan_cat_cols`, but use the string `"passthrough"` instead of `StandardScaler()` for `loan_num_cols`.
2. `fit_transform` it on `loan_X_train`, and `transform` on `loan_X_test`.
3. Train a new `KNeighborsClassifier(n_neighbors=5)` on this unscaled data.
4. Print its accuracy with `accuracy_score`, next to `loan_baseline_acc` for comparison.

**Observe:** Accuracy should drop. In a comment, write one line answering: *why can a large-scale column affect KNN?*

---



In [ ]:
# Write your code here





## **Q1.2 — Try Different k**

**Concept:** `k` is just a setting (hyperparameter) we chose — we assumed `k = 5` without checking if it's actually the best choice.

**Your task:** Using the already-scaled `loan_X_train_t` / `loan_X_test_t` from Setup, write a loop that:
1. Starts an empty list called `loan_accuracies`.
2. Loops `k` over `range(1, 30, 2)` (every odd number).
3. Inside the loop: creates a `KNeighborsClassifier(n_neighbors=k)`, fits it on `loan_X_train_t` / `loan_y_train`, and appends its accuracy on the test set to `loan_accuracies`.

Run the provided cell below yours to see the accuracy-vs-k graph and the best `k`.

---



In [ ]:
# Write your code here





### _Provided — run this after your loop above to see the graph and the best `k`._

---



In [ ]:
sns.lineplot(x=list(range(1, 30, 2)), y=loan_accuracies, marker='o')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.title('Loan Approval: Accuracy vs k')
plt.show()

best_k = list(range(1, 30, 2))[loan_accuracies.index(max(loan_accuracies))]
print('Best k:', best_k, '-> accuracy:', max(loan_accuracies))


## **Q1.3 — Pipeline**

**Concept:** A `Pipeline` combines preprocessing and the model into one object, so you don't have to transform your data and fit your model as two separate steps every time.

**Your task:** You already have a working, scaled preprocessor — `loan_ct` from Setup. Reuse it directly, you do not need to rebuild it.
1. Build a `Pipeline` named `pipe` with two steps: `'preprocess'` → `loan_ct`, and `'knn'` → `KNeighborsClassifier(n_neighbors=best_k)`.
2. Fit `pipe` on `loan_X_train` and `loan_y_train` directly (raw, untransformed data — the Pipeline handles the transforming for you).
3. Print its accuracy on `loan_X_test` / `loan_y_test`, and confirm it matches your best accuracy from Q1.2.

---



In [ ]:
# Write your code here





# ✈️ **Part 2 — Flight Delay (HOMEWORK / MINI-CAPSTONE)**

### _Apply the exact same ideas from Part 1 — on your own, in small steps — using `flight_delay.csv`. Every step below is still only 3–5 lines of your own code._

---



### _Provided — run this first to load and inspect the data._

---



In [ ]:
flight_df = pd.read_csv('flight_delay.csv')
print(flight_df.shape)
print(flight_df.isnull().sum())


## **Q2.1 — Clean the Data**

**Your task:** Looking at the output above, drop the ID column, and fill the two numeric columns that have missing values with their median — same pattern you used for `loan_df` in Setup.

---



In [ ]:
# Write your code here





## **Q2.2 — Features, Target, and Split**

**Your task:**
1. Create `X` (drop the `delayed` column) and `y` (the `delayed` column) from `flight_df`.
2. Split them into `X_train`, `X_test`, `y_train`, `y_test` — 80/20, `random_state=42`.

### _Provided — the column lists for the next step:_

---



In [ ]:
flight_cat_cols = ['airline', 'weather', 'day_type']
flight_num_cols = ['departure_hour', 'airport_traffic', 'plane_age_years', 'distance_km']


In [ ]:
# Write your code here





## **Q2.3 — Preprocessing + Pipeline**

**Your task:**
1. Build a `ColumnTransformer` called `flight_pre` — `OneHotEncoder` on `flight_cat_cols`, `StandardScaler` on `flight_num_cols`.
2. Build a `Pipeline` called `flight_pipe` with `flight_pre` and a `KNeighborsClassifier(n_neighbors=5)`.
3. Fit it on `X_train` / `y_train`, and print its accuracy on `X_test` / `y_test`.

---



In [ ]:
# Write your code here





## **Q2.4 — Try Different k**

**Your task:** Same loop pattern as Q1.2, but build a fresh `Pipeline` inside the loop each time (reusing `flight_pre`), for every odd `k` from 1 to 29. Store each accuracy in a list called `flight_accuracies`.

Run the provided cell below yours to see the graph and the best `k`.

---



In [ ]:
# Write your code here





### _Provided — run this after your loop above._

---



In [ ]:
sns.lineplot(x=list(range(1, 30, 2)), y=flight_accuracies, marker='o', color='purple')
plt.xlabel('k')
plt.ylabel('Accuracy')
plt.title('Flight Delay: Accuracy vs k')
plt.show()

best_k_flight = list(range(1, 30, 2))[flight_accuracies.index(max(flight_accuracies))]
print('Best k:', best_k_flight, '-> accuracy:', max(flight_accuracies))


## **Q2.5 — Write Up Your Result**

**Your task:** Fill in the three comments below with your own findings.

---



In [ ]:
# Best k: ____
# Accuracy at best k: ____
# If I skipped scaling, the column that would hurt the model most is ____, because ____
